In [ ]:
import torch, detectron2
!nvcc --version
TORCH_VERSION = ".".join(torch.__version__.split(".")[:2])
CUDA_VERSION = torch.__version__.split("+")[-1]
print("torch: ", TORCH_VERSION, "; cuda: ", CUDA_VERSION)
print("torch built with cuda:", torch.cuda.is_available())
print("detectron2:", detectron2.__version__)

In [ ]:
# Some basic setup:
# Setup detectron2 logger
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

# import some common detectron2 utilities
from detectron2.config import get_cfg
from detectron2.structures import BoxMode
from detectron2.data import MetadataCatalog, DatasetCatalog, DatasetMapper
from detectron2.data import transforms as T
from detectron2.data import detection_utils, build_detection_train_loader
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.projects.point_rend import add_pointrend_config
from detectron2.utils.visualizer import Visualizer
from detectron2.evaluation import COCOEvaluator

# import some common libraries
import numpy as np
import os
import cv2
from PIL import Image
import pycocotools
from pycocotools import mask as coco_mask_util
import pickle
import copy
from typing import Dict

## Dataset
### Preparing the datasets
In the following, we read the train/test datasets and convert them to the lightweight dictionary format expected by the Detectron2 dataloader. Since this reading/parsing part takes a while, we save the results to disk and load them from disk in the function used for loading our custom datasets. 

Run the following cells only once for any new dataset to use. Jump to "Registering the datasets" step.

In [ ]:
def parse_cell_mask_dataset(images_path: str, 
                            masks_path: str, 
                            percentage_to_expand_bbox_boundaries: float=0.0, 
                            annots_in_coco_rle_format: bool=True):
    
    # list of images and mask annotation files
    # image_xyz.jpg in the images_path folder should have a corresponding annotation file
    # with the same name and npz format, image_xyz.npz in masks_path folder
    img_filenames = list(sorted(os.listdir(images_path)))
    mask_filenames = list(sorted(os.listdir(masks_path)))
    
    dataset_dicts: list[dict] = []
    # going though the images
    for idx, img_filename in enumerate(img_filenames):
        
        # a dictionary to record the annotations for the image under consideration
        record: dict = {}
        
        # drop the image filename extensio, the mask filename should have the same
        # name as the image with either 'npz' or 'pkl' as the extension
        # (anything after the last '.' in the filename is considered as extension)
        img_name = ".".join(img_filename.strip().split('.')[:-1])
    
        # load the image
        img_path = os.path.join(images_path, img_filename)
        
        # read the image, these images are pre-processed in grayscale with bit-depth = 8
        # no further scaling is needed here
        img = Image.open(img_path)
        image_width, image_height = img.size
        
        if annots_in_coco_rle_format:
            # annotations are in COCO RLE format
            
            boxes: List[List[int]] = []
            labels: List[int] = []
            masks: List[np.ndarray] = []
            
            # load the annotations (pickled when annots_in_coco_rle_format is True)
            mask_filename = img_name + '.pkl'
            if mask_filename not in mask_filenames:
                # skip if the annotation does not exist
                continue
            
            mask_path = os.path.join(masks_path, mask_filename)
            filehandler = open(mask_path, 'rb')
            annots = pickle.load(filehandler)
            filehandler.close()
            
            for sample in annots['annotations']:
                xmin, ymin, xmax, ymax = sample['bbox']
                # no need to check the validity 
                if xmin >= xmax or ymin >= ymax:
                    continue
                
                labels.append(int(sample['category_id']))
                mask: np.ndarray = coco_mask_util.decode(sample['segmentation'])
                # mask in full image resolution
                full_res_mask: np.ndarray = np.zeros((image_height, image_width), np.uint8)
                full_res_mask[ymin:ymax, xmin:xmax] = mask
                
                masks.append(full_res_mask)
                
                # expand the bounding box if needed
                delta_x = int(percentage_to_expand_bbox_boundaries * (xmax - xmin) / 2)
                delta_y = int(percentage_to_expand_bbox_boundaries * (ymax - ymin) / 2)
            
                delta_x = max(1, delta_x)
                delta_y = max(1, delta_y)
            
                xmin = max(0, xmin - delta_x)
                ymin = max(0, ymin - delta_y)
                xmax = min(image_width, xmax + delta_x)
                ymax = min(image_height, ymax + delta_y)
                
                boxes.append([xmin, ymin, xmax, ymax])
                
            num_objs = len(boxes)
            # combine all the masks
            masks = np.array(masks)
            labels = np.array(labels)
        else:
            # loaded_mask is a m x image_height x image_width array (the same size as the input image)
            # the assumption here is instances are encoded as different levels
            # with 0 being the background
            # each element in mask is an np.uint16 (unsigned 16 bits)
            # to support overlapping objects, objects with overlaps are reported in 
            # different arrays (loaded_mask[i])
            # so we simply need to extract masks from all these arrays
            mask_filename = img_name + '.npz'
            if mask_filename not in mask_filenames:
                continue
            
            mask_path = os.path.join(masks_path, mask_filename)
            loaded = np.load(mask_path)
            loaded_masks = loaded['saved_masks']
            loaded_labels = loaded['saved_labels']
        
            num_objs = 0
            labels: List[int] = []
            masks: List[np.ndarray] = []
            
            for i in range(loaded_masks.shape[0]):
                # instances are encoded as different gray levels
                # the results are sorted
                # first id [0] is the background, so remove it
                obj_ids = np.unique(loaded_masks[i])[1:]
        
                num_objs += len(obj_ids)
        
                # split the color-encoded mask into a set
                # of binary masks
                masks.append((loaded_masks[i] == obj_ids[:, None, None]).astype(np.uint8))
                # make sure labels are mapped correctly to masks
                labels += list(loaded_labels[obj_ids - 1])
        
            # combine all the masks
            masks = np.concatenate(masks, axis=0)
            labels = np.array(labels)
        
            # get bounding box coordinates for each mask
            boxes: List[List[int]] = []
            valid_ids = []
            for i in range(num_objs):
                pos = np.where(masks[i])
                xmin = np.min(pos[1])
                xmax = np.max(pos[1])
                ymin = np.min(pos[0])
                ymax = np.max(pos[0])
            
                if xmin >= xmax or ymin >= ymax:
                    continue
            
                delta_x = int(percentage_to_expand_bbox_boundaries * (xmax - xmin) / 2)
                delta_y = int(percentage_to_expand_bbox_boundaries * (ymax - ymin) / 2)
            
                delta_x = max(1, delta_x)
                delta_y = max(1, delta_y)
            
                xmin = max(0, xmin - delta_x)
                ymin = max(0, ymin - delta_y)
                xmax = min(image_width, xmax + delta_x)
                ymax = min(image_height, ymax + delta_y)
            
                valid_ids.append(i)
                boxes.append([xmin, ymin, xmax, ymax])
        
            num_objs = len(valid_ids)
            labels = labels[valid_ids]
            masks = masks[valid_ids]
        
        record['file_name'] = img_path
        record['image_id'] = idx
        record['height'] = image_height
        record['width'] = image_width 
        record['annotations']: List[dict] = []
        # going through the annotated objects in the image under consideration
        for i in range(num_objs):
            annots: dict = {'bbox': [boxes[i][0], boxes[i][1], boxes[i][2], boxes[i][3]],
                            'bbox_mode': BoxMode.XYXY_ABS,
                            'category_id' : int(labels[i] - 1), # labels start from 0 in Detectron
                            'segmentation': coco_mask_util.encode(np.asarray(masks[i], order="F")),
                             'iscrowd': 0
                           } 
                        # cfg.INPUT.MASK_FORMAT must be set to bitmask if using the default data loader 
                        # with dict 'segmentation mask above'.

            record['annotations'].append(annots)
        # end for
        dataset_dicts.append(record)
    # end for
    return dataset_dicts

#### Reading the datasets
This will take a while. 

In [ ]:
# make sure each folder includes both 'train' and 'test' sub-folders containing the images
# and masks for the train/test sets
IMAGES_FOLDER = '/home/cellareye/Cellanome/dl-mehdi/Mask RCNN/data/analysis_data_cells_set_2_crop_2/images/'
MASKS_FOLDER = '/home/cellareye/Cellanome/dl-mehdi/Mask RCNN/data/analysis_data_cells_set_2_crop_2/masks/'

import time
start = time.time()
dataset_type = 'test'
dataset_dicts_test = parse_cell_mask_dataset(images_path = IMAGES_FOLDER + dataset_type, 
                                             masks_path = MASKS_FOLDER + dataset_type, 
                                             percentage_to_expand_bbox_boundaries=0.1, 
                                             annots_in_coco_rle_format=True)


print("Reading test dataset took: ", time.time() - start)

start = time.time()
dataset_type = 'train'
dataset_dicts_train = parse_cell_mask_dataset(images_path = IMAGES_FOLDER + dataset_type, 
                                              masks_path = MASKS_FOLDER + dataset_type, 
                                              percentage_to_expand_bbox_boundaries=0.1, 
                                              annots_in_coco_rle_format=True)
print("Reading train dataset took: ", time.time() - start)

#### Saving the datasets to disk

In [ ]:
filehandler = open('data/test_data.pkl', 'wb')
pickle.dump(dataset_dicts_test, filehandler)
filehandler.close()

filehandler = open('data/train_data.pkl', 'wb')
pickle.dump(dataset_dicts_train, filehandler)
filehandler.close()

### Registering the datasets
The following custom dataset reader function only loads the already processed data from disk. 

In [ ]:
def get_cell_mask_dataset(dataset_type: str):
    if dataset_type == 'train':
        filehandler = open('data/train_data.pkl', 'rb')
    elif dataset_type == 'test':
        filehandler = open('data/test_data.pkl', 'rb')
    else:
        return []
    
    dataset_dicts = pickle.load(filehandler)
    filehandler.close()
    return dataset_dicts

In [ ]:
for dataset_type in ['train', 'test']:
    DatasetCatalog.register('analysis_data_cells_set_2_crop_2_' + dataset_type, 
                            lambda dataset_type=dataset_type: get_cell_mask_dataset(dataset_type))
    
    MetadataCatalog.get('analysis_data_cells_set_2_crop_2_' + dataset_type).set(thing_classes=['cell', 'bead', 'cage'])
    MetadataCatalog.get('analysis_data_cells_set_2_crop_2_' + dataset_type).set(evaluator_type='coco')


## Model Definition
We use the PointRend model with ResNet50 + FPN from Detectron2 that has been trained on COCO dataset (80 classes) as the starting point and modify its head. 

In [ ]:
# get detectron2's default config
cfg = get_cfg()
# add PointRend-specific default config
add_pointrend_config(cfg)
# load the RCNN ResNet50 + FPN PointRend model config from file
cfg.merge_from_file("/home/cellareye/Development/detectron2/projects/PointRend/configs/InstanceSegmentation/pointrend_rcnn_R_50_FPN_3x_coco.yaml")
# this threshold is needed for inference
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5  # set threshold for this model

# use a model from PointRend model zoo: https://github.com/facebookresearch/detectron2/tree/master/projects/PointRend#pretrained-models
cfg.MODEL.WEIGHTS = "detectron2://PointRend/InstanceSegmentation/pointrend_rcnn_R_50_FPN_3x_coco/164955410/model_final_edd263.pkl"

#  NOTE: this is the number of classes, num_classes and not num_classes+1
cfg.MODEL.POINT_HEAD.NUM_CLASSES = 3
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 3

# anchors, we set this similar to our Mask-RCNN setting
cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[12], [24], [36], [48], [60]]

# increase the number of proposals to keep before applying NMS and after
# applying NMS during training and testing
# we target for 500 cells in an image, so we need to make sure
# enough region proposals are considered specially during testing/eval
# (default values for both pre and post are 2000 and 1000 for training
# and testing, respectively)
cfg.MODEL.RPN.PRE_NMS_TOPK_TRAIN = 8000
cfg.MODEL.RPN.PRE_NMS_TOPK_TEST = 4000
cfg.MODEL.RPN.POST_NMS_TOPK_TRAIN = 8000
cfg.MODEL.RPN.POST_NMS_TOPK_TEST = 4000

# increase the total number of anchors (positive and negative) that are 
# sampled during training of RPN (for computing loss, default is 256; by 
# default 0.5 will be positive anchors)
cfg.MODEL.RPN.BATCH_SIZE_PER_IMAGE = 1024

# increase the total number of anchors (positive and negative) that are 
# sampled during training of classification head (for computing loss,
# default is 512; by default 0.25 will be positive anchors)
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 2048


# already set correctly
# cfg.INPUT.MASK_FORMAT = 'bitmask'

## Training
### Datasets

In [ ]:
cfg.DATASETS.TRAIN = ('analysis_data_cells_set_2_crop_2_train',)
cfg.DATASETS.TEST = ('analysis_data_cells_set_2_crop_2_test',)
# evaluation period (number of iterations)
cfg.TEST.EVAL_PERIOD = 10000
cfg.DATALOADER.NUM_WORKERS = 32

### Check some samples

In [ ]:
def show_sample(idx, train=True, mapper=None):
    if train:
        dataset_type = 'train'
        registered_dataset = cfg.DATASETS.TRAIN[0]
    else:
        dataset_type = 'test'
        registered_dataset = cfg.DATASETS.TEST[0]

    dataset_dicts = get_cell_mask_dataset(dataset_type)
    analysis_data_cells_metadata = MetadataCatalog.get(registered_dataset)
    
    sample = dataset_dicts[idx]
    if mapper is None: 
        img = cv2.imread(sample['file_name'])
        print("Image shape: ", img.shape)
        visualizer = Visualizer(img[:, :, ::-1], metadata=analysis_data_cells_metadata, scale=0.5)
        out = visualizer.draw_dataset_dict(sample)
    else:
        transformed_sample = mapper(sample)
        img = np.array(transformed_sample["image"]).transpose((1, 2, 0))
        print("Image shape: ", img.shape)
        visualizer = Visualizer(img[:, :, ::-1], metadata=analysis_data_cells_metadata, scale=0.5)
        out = visualizer.overlay_instances(
            boxes=transformed_sample["instances"].gt_boxes, 
            masks=transformed_sample["instances"].gt_masks, 
            labels=[analysis_data_cells_metadata.thing_classes[i - 1] for i in 
                    np.array(transformed_sample["instances"].gt_classes)])

    display(Image.fromarray(out.get_image()[:, :, ::-1]))

In [ ]:
show_sample(idx=40, train=False)

### Augmentations
For now, we are using mainly random resizing (0.8-1).

In [ ]:
AUGMENTATIONS = [
        T.RandomBrightness(0.8, 1.2),
        T.RandomFlip(prob=0.5),
        T.RandomResize([(820, 640),
                        (844, 660),
                        (870, 680),
                        (896, 700),
                        (920, 720),
                        (948, 740),
                        (972, 760),
                        (998, 780),
                        (1024, 800)])
]

#### Check the sample after transformation

In [ ]:
# we are using the below mapper for displaying augmented samples, later we define it in the training class
mapper = DatasetMapper(cfg, is_train=True, augmentations=AUGMENTATIONS)

In [ ]:
show_sample(idx=40, train=False, mapper=mapper)

### Training parameters

In [ ]:
# this is the batch size 
cfg.SOLVER.IMS_PER_BATCH = 2  
cfg.SOLVER.BASE_LR = 0.005  # initial Learning rate
cfg.SOLVER.MAX_ITER = 80000 # number of iterations
cfg.SOLVER.STEPS = [25000, 62000, 70000] # steps      
cfg.freeze()

### Customized Trainer class

In [ ]:
class Trainer(DefaultTrainer):
    """
    We use the "DefaultTrainer" which contains a number pre-defined logic for
    standard training workflow. They may not work for you, especially if you
    are working on a new research project. In that case you can use the cleaner
    "SimpleTrainer", or write your own training loop.
    """

    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        """
        Create evaluator(s) for a given dataset.
        This uses the special metadata "evaluator_type" associated with each builtin dataset.
        For your own dataset, you can simply create an evaluator manually in your
        script and do not have to worry about the hacky if-else logic here.
        """
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "inference", dataset_name)
        
        return COCOEvaluator(dataset_name, 
                             tasks = ['bbox', 'segm'], 
                             max_dets_per_image = 1000, 
                             output_dir=output_folder)
    
    @classmethod
    def build_train_loader(cls, cfg):
        mapper = DatasetMapper(cfg, is_train=True, augmentations=AUGMENTATIONS)
        return build_detection_train_loader(cfg, mapper=mapper)
    

### Run training for the specific number of iterations

In [ ]:
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
trainer = Trainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

In [ ]:
"""
Evaluate annotation type *bbox*
DONE (t=81.35s).
Accumulating evaluation results...
DONE (t=0.64s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=1000 ] = 0.670
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=1000 ] = 0.830
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=1000 ] = 0.774
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=1000 ] = 0.431
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=1000 ] = 0.590
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=1000 ] = 0.957
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.091
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.408
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=1000 ] = 0.702
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1000 ] = 0.469
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=1000 ] = 0.672
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= large | maxDets=1000 ] = 0.973
[02/08 21:22:12 d2.evaluation.coco_evaluation]: Evaluation results for bbox: 
|   AP   |  AP50  |  AP75  |  APs   |  APm   |  APl   |
|:------:|:------:|:------:|:------:|:------:|:------:|
| 67.008 | 83.009 | 77.401 | 43.148 | 58.976 | 95.685 |
[02/08 21:22:12 d2.evaluation.coco_evaluation]: Per-category bbox AP: 
| category   | AP     | category   | AP     | category   | AP     |
|:-----------|:-------|:-----------|:-------|:-----------|:-------|
| cell       | 56.246 | bead       | 51.244 | cage       | 93.534 |

Evaluate annotation type *segm*
DONE (t=82.19s).
Accumulating evaluation results...
DONE (t=0.65s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=1000 ] = 0.677
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=1000 ] = 0.830
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=1000 ] = 0.775
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=1000 ] = 0.442
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=1000 ] = 0.634
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=1000 ] = 0.956
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.091
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.409
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=1000 ] = 0.707
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1000 ] = 0.479
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=1000 ] = 0.670
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= large | maxDets=1000 ] = 0.973
[02/08 21:23:37 d2.evaluation.coco_evaluation]: Evaluation results for segm: 
|   AP   |  AP50  |  AP75  |  APs   |  APm   |  APl   |
|:------:|:------:|:------:|:------:|:------:|:------:|
| 67.730 | 83.010 | 77.490 | 44.184 | 63.395 | 95.576 |
[02/08 21:23:37 d2.evaluation.coco_evaluation]: Per-category segm AP: 
| category   | AP     | category   | AP     | category   | AP     |
|:-----------|:-------|:-----------|:-------|:-----------|:-------|
| cell       | 56.867 | bead       | 52.611 | cage       | 93.711 |
"""

### Test set evaluation (COCO metrics)

In [ ]:
from collections import OrderedDict
import detectron2.utils.comm as comm
from tools.plain_train_net import build_model, build_detection_test_loader, inference_on_dataset, print_csv_format
import logging
logger = logging.getLogger("detectron2")

In [ ]:
def do_evaluate(cfg, model):
    results = OrderedDict()
    for dataset_name in cfg.DATASETS.TEST:
        data_loader = build_detection_test_loader(cfg, dataset_name)

        evaluator = COCOEvaluator(dataset_name, 
                                  tasks = ['bbox', 'segm'], 
                                  max_dets_per_image = 1000, 
                                  output_dir=os.path.join(cfg.OUTPUT_DIR, "inference", dataset_name))  
        results_i = inference_on_dataset(model, data_loader, evaluator)
        results[dataset_name] = results_i
        if comm.is_main_process():
            logger.info("Evaluation results for {} in csv format:".format(dataset_name))
            print_csv_format(results_i)
    if len(results) == 1:
        results = list(results.values())[0]
    return results

In [ ]:
# get detectron2's default config
cfg = get_cfg()
# add PointRend-specific default config
add_pointrend_config(cfg)
# load a specific PointRend config from file
cfg.merge_from_file("/home/cellareye/Development/detectron2/projects/PointRend/configs/InstanceSegmentation/pointrend_rcnn_R_50_FPN_3x_coco.yaml")
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5  # set threshold for this model (needed for inference)
cfg.DATASETS.TRAIN = ('analysis_data_cells_set_2_crop_2_train',)
cfg.DATASETS.TEST = ('analysis_data_cells_set_2_crop_2_test',)
cfg.DATALOADER.NUM_WORKERS = 32
#  NOTE: this is the number of classes, num_classes and not num_classes+1
cfg.MODEL.POINT_HEAD.NUM_CLASSES = 3
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 3

# anchors, we set this similar to our Mask-RCNN setting
cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[12], [24], [36], [48], [60]]

# increase the number of proposals to keep before applying NMS and after
# applying NMS during training and testing
# we target for 500 cells in an image, so we need to make sure
# enough region proposals are considered specially during testing/eval
# (default values for both pre and post are 2000 and 1000 for training
# and testing, respectively)
cfg.MODEL.RPN.PRE_NMS_TOPK_TRAIN = 8000
cfg.MODEL.RPN.PRE_NMS_TOPK_TEST = 4000
cfg.MODEL.RPN.POST_NMS_TOPK_TRAIN = 8000
cfg.MODEL.RPN.POST_NMS_TOPK_TEST = 4000

# increase the total number of anchors (positive and negative) that are 
# sampled during training of RPN (for computing loss, default is 256; by 
# default 0.5 will be positive anchors)
cfg.MODEL.RPN.BATCH_SIZE_PER_IMAGE = 1024

# increase the total number of anchors (positive and negative) that are 
# sampled during training of classification head (for computing loss,
# default is 512; by default 0.25 will be positive anchors)
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 2048
# already set correctly
cfg.INPUT.MASK_FORMAT = 'bitmask'
# my trained model
cfg.MODEL.WEIGHTS = "/home/cellareye/Cellanome/dl-mehdi/PointRend/output/model_final.pth"
cfg.freeze()

In [ ]:
model = build_model(cfg)

In [ ]:
do_evaluate(cfg, model)

### Inference

In [ ]:
IMAGES_FOLDER = '/home/cellareye/Cellanome/dl-mehdi/Mask RCNN/data/analysis_data_cells_set_2_crop_2/images/'
img = cv2.imread(IMAGES_FOLDER + 'test/1_1_1_83_1_069000_048512_-00087_BF (TI)_crp_17.jpg')
predictor = DefaultPredictor(cfg)
outputs = predictor(img)
# We can use `Visualizer` to draw the predictions on the image.
v = Visualizer(img[:, :, ::-1], MetadataCatalog.get(cfg.DATASETS.TRAIN[0]), scale=1.2)
out = v.draw_instance_predictions(outputs["instances"].to("cpu"))
Image.fromarray(out.get_image()[:, :, ::-1])